**Create Cognitive Reframing dataset splits**

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# Paths
DATA_IN = Path("../../processed/cognitive_reframing.csv")
OUT_DIR = Path("../../processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
RANDOM_SEED = 42
TEST_SIZE = 0.20

In [2]:
df_raw = pd.read_csv(DATA_IN)
print(f"Raw shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(f"\nUnique thoughts: {df_raw['thought_id'].nunique()} "
      f"(in {len(df_raw)} rows — {len(df_raw) / df_raw['thought_id'].nunique():.2f} reframes/thought)")
print(f"\ndominant_label distribution:")
print(df_raw["dominant_label"].value_counts())

Raw shape: (600, 9)
Columns: ['thought_id', 'situation', 'thought', 'reframe', 'dominant_label', 'all_labels', 'situation_length', 'thought_length', 'reframe_length']

Unique thoughts: 286 (in 600 rows — 2.10 reframes/thought)

dominant_label distribution:
dominant_label
negative feeling or emotion    80
labeling                       70
overgeneralizing               61
mind reading                   57
all-or-nothing thinking        57
fortune telling                56
personalizing                  55
disqualifying the positive     35
catastrophizing                35
emotional reasoning            31
none                           18
should statements              18
blaming                        17
comparing and despairing        8
magnification                   2
Name: count, dtype: int64


In [3]:
df = df_raw.copy()

# Use dominant_label as the category (already the first label, per the dataset's design)
df["category"] = df["dominant_label"]

# Rule: drop 'none' rows entirely
n_before = len(df)
n_thoughts_before = df["thought_id"].nunique()
df = df[df["category"] != "none"].reset_index(drop=True)

# Drop 'magnification': only 1 unique thought, cannot be stratified
df = df[df["category"] != "magnification"].reset_index(drop=True)

n_after = len(df)
n_thoughts_after = df["thought_id"].nunique()

print(f"Dropped 'none': {n_before} -> {n_after} rows  "
      f"({n_thoughts_before} -> {n_thoughts_after} unique thoughts)")
print(f"\nFinal category distribution:")
print(df["category"].value_counts())
print(f"\nNumber of distinct categories: {df['category'].nunique()}")

Dropped 'none': 600 -> 580 rows  (286 -> 283 unique thoughts)

Final category distribution:
category
negative feeling or emotion    80
labeling                       70
overgeneralizing               61
mind reading                   57
all-or-nothing thinking        57
fortune telling                56
personalizing                  55
disqualifying the positive     35
catastrophizing                35
emotional reasoning            31
should statements              18
blaming                        17
comparing and despairing        8
Name: count, dtype: int64

Number of distinct categories: 13


**Check feasibility of thought-level stratified split**

The split is now at the thought_id level, so the stratification key is the category of each unique thought. Each thought_id has a single dominant_label (the dataset is consistent on this), so this is well-defined.

In [4]:
# One row per unique thought, with its category
thought_level = df.drop_duplicates(subset="thought_id")[["thought_id", "category"]].reset_index(drop=True)

print(f"Total unique thoughts to split: {len(thought_level)}")
print(f"\nThought-level category counts:")
print(thought_level["category"].value_counts())

cat_counts = thought_level["category"].value_counts()
singletons = cat_counts[cat_counts < 2]
small_cats = cat_counts[cat_counts < 5]

print(f"\nCategories with <2 unique thoughts (cannot stratify): {len(singletons)}")
if len(singletons) > 0:
    print(singletons)
print(f"\nCategories with <5 unique thoughts (test set will have 0-1 examples):")
print(small_cats)

Total unique thoughts to split: 283

Thought-level category counts:
category
all-or-nothing thinking        39
negative feeling or emotion    37
labeling                       36
mind reading                   33
catastrophizing                25
fortune telling                22
emotional reasoning            18
overgeneralizing               17
disqualifying the positive     17
personalizing                  16
blaming                        10
should statements               7
comparing and despairing        6
Name: count, dtype: int64

Categories with <2 unique thoughts (cannot stratify): 0

Categories with <5 unique thoughts (test set will have 0-1 examples):
Series([], Name: count, dtype: int64)


In [5]:
# train_ids and test_ids are two arrays of thought_id values
train_ids, test_ids = train_test_split(
    thought_level["thought_id"].values,
    test_size=TEST_SIZE,
    stratify=thought_level["category"].values,
    random_state=RANDOM_SEED,
)
# go back to the full row-level dataframe df
# (where each thought may appear in multiple rows because
# of multiple reframes per thought) and pull out the matching rows:

# Produce a boolean mask; a Series of True/False the same length as df,
# where each entry is True if that row's thought_id is in train_ids, and False otherwise.
# train_df is every row whose thought_id belongs to the training split
train_df = df[df["thought_id"].isin(train_ids)].reset_index(drop=True)
# test_df is every row whose thought_id belongs to the test split
test_df = df[df["thought_id"].isin(test_ids)].reset_index(drop=True)

print(f"Train (dev pool): {len(train_df)} rows, {train_df['thought_id'].nunique()} unique thoughts")
print(f"Test (held-out):  {len(test_df)} rows, {test_df['thought_id'].nunique()} unique thoughts")

Train (dev pool): 467 rows, 226 unique thoughts
Test (held-out):  113 rows, 57 unique thoughts


In [6]:
# Distribution check
train_dist = train_df.drop_duplicates("thought_id")["category"].value_counts(normalize=True).sort_index()
test_dist = test_df.drop_duplicates("thought_id")["category"].value_counts(normalize=True).sort_index()

dist_check = pd.DataFrame({
    "train_pct": (train_dist * 100).round(2),
    "test_pct": (test_dist * 100).round(2),
    "train_thoughts": train_df.drop_duplicates("thought_id")["category"].value_counts().sort_index(),
    "test_thoughts": test_df.drop_duplicates("thought_id")["category"].value_counts().sort_index(),
}).fillna(0)

print("Thought-level distribution (proportions should be ~equal):")
print(dist_check)

Thought-level distribution (proportions should be ~equal):
                             train_pct  test_pct  train_thoughts  \
category                                                           
all-or-nothing thinking          13.72     14.04              31   
blaming                           3.54      3.51               8   
catastrophizing                   8.85      8.77              20   
comparing and despairing          2.21      1.75               5   
disqualifying the positive        6.19      5.26              14   
emotional reasoning               6.19      7.02              14   
fortune telling                   7.52      8.77              17   
labeling                         12.83     12.28              29   
mind reading                     11.50     12.28              26   
negative feeling or emotion      12.83     14.04              29   
overgeneralizing                  6.19      5.26              14   
personalizing                     5.75      5.26         

In [7]:
# Leakage check at thought_id level (should be impossible by construction, but verify)
train_ids_set = set(train_df["thought_id"])
test_ids_set = set(test_df["thought_id"])
id_overlap = train_ids_set & test_ids_set

# Also verify at thought-text level as a belt-and-braces check
train_thoughts = set(train_df["thought"].str.strip().str.lower())
test_thoughts = set(test_df["thought"].str.strip().str.lower())
text_overlap = train_thoughts & test_thoughts

print(f"thought_id overlap:   {len(id_overlap)} (must be 0)")
print(f"thought-text overlap: {len(text_overlap)} (should be 0)")
assert len(id_overlap) == 0, "thought_id leakage detected"

thought_id overlap:   0 (must be 0)
thought-text overlap: 0 (should be 0)


In [8]:
train_path = OUT_DIR / "cognitive_reframing_train.csv"
test_path = OUT_DIR / "cognitive_reframing_test.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Saved train: {train_path}")
print(f"Saved test:  {test_path}")
print(f"\nColumns saved: {list(train_df.columns)}")

Saved train: ../../processed/cognitive_reframing_train.csv
Saved test:  ../../processed/cognitive_reframing_test.csv

Columns saved: ['thought_id', 'situation', 'thought', 'reframe', 'dominant_label', 'all_labels', 'situation_length', 'thought_length', 'reframe_length', 'category']


In [9]:
train_reloaded = pd.read_csv(train_path)
test_reloaded = pd.read_csv(test_path)

assert len(train_reloaded) == len(train_df), "Train round-trip failed"
assert len(test_reloaded) == len(test_df), "Test round-trip failed"
assert set(train_reloaded["category"]) == set(train_df["category"]), "Train category mismatch"

print(f"Train reloaded: {len(train_reloaded)} rows")
print(f"Test reloaded:  {len(test_reloaded)} rows")
print(f"\nSample test row:")
print(test_reloaded.iloc[0].to_dict())

Train reloaded: 467 rows
Test reloaded:  113 rows

Sample test row:
{'thought_id': 3, 'situation': 'A neighbor borrowed my smartphone charger and never returned it.', 'thought': 'he took me for granted!', 'reframe': 'I know my neighbor is busy with his job and family. He probably just forgot to return it to me. I should stop by and ask for it back.', 'dominant_label': 'labeling', 'all_labels': '["labeling", "catastrophizing"]', 'situation_length': 64, 'thought_length': 23, 'reframe_length': 133, 'category': 'labeling'}
